In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold
from transformers import BertTokenizer
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset,random_split,SubsetRandomSampler, ConcatDataset
from transformers import BertModel
from torch.optim import Adam
from tqdm import tqdm
from torchmetrics.classification import BinaryStatScores

## Switching to CUDA

In [2]:
# Get cpu or gpu device for training.
use_cuda = torch.cuda.is_available()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


## Loading Dataset

In [6]:
df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/DataFormated4BERT.pkl')
df = df.loc[((df['Party'] == 'R')|(df['Party'] == 'D'))&(df['is_question'] == True)]
D = len(df.loc[df['Party']=='D'])
R = len(df.loc[df['Party']=='R'])
print(
    f'\nTotal number of utterances: {len(df)}\
    \nTotal number of Democrat utterances: {D}, {D/len(df): .2f}%\
    \nTotal number of Republicam utterances: {R}, {R/len(df): .2f}%'
)


Total number of utterances: 611651    
Total number of Democrat utterances: 296793, 0.4852325917884545%    
Total number of Republicam utterances: 314858, 0.5147674082115454%


## Datset Class

In [7]:
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
labels_dict = {'R':0,'D':1}

class CongressHearings(Dataset):

    def __init__(self, df, isPredict = False):

        if isPredict:
            self.labels = [int(label) for label in df['Party']]
        else:
            self.labels = [labels_dict[label] for label in df['Party']]
        
        self.texts = [tokenizer(str(text), padding='max_length', max_length = 512, truncation=True, return_tensors="pt") for text in df['text_without_stopwords']]
        

    def classes(self):
        return self.labels

    def __len__(self):
        return len(self.labels)

    def get_batch_labels(self, idx):
        # Fetch a batch of labels
        return np.array(self.labels[idx])

    def get_batch_texts(self, idx):
        # Fetch a batch of inputs
        return self.texts[idx]

    def __getitem__(self, idx):
        
        batch_texts = self.get_batch_texts(idx)
        batch_y = self.get_batch_labels(idx)

        return batch_texts, batch_y


## Building the Model

In [9]:
class BertClassifier(nn.Module):

    def __init__(self, dropout=0.5):

        super(BertClassifier, self).__init__()

        self.bert = BertModel.from_pretrained('bert-base-cased')
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(768, 2)
        self.relu = nn.ReLU()

    def forward(self, input_id, mask):

        _, pooled_output = self.bert(input_ids= input_id, attention_mask=mask,return_dict=False)
        dropout_output = self.dropout(pooled_output)
        linear_output = self.linear(dropout_output)
        final_layer = self.relu(linear_output)

        return final_layer

## Loading the Model

In [10]:
model = BertClassifier()
model.load_state_dict(torch.load('/mnt/fstore/DataFiles/QnA/Models/PartyBERT.pth'))
model.eval()

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tru

## Testing the saved Model

In [11]:
def evaluate(model, test):

    test_dataloader = DataLoader(test, batch_size=1)

    y_pred = torch.tensor([])
    y_true = torch.tensor([])

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    y_pred = y_pred.to(device)
    y_true = y_true.to(device)

    if use_cuda:

        model = model.cuda()

    total_acc_test = 0
    with torch.no_grad():

        for test_input, test_label in test_dataloader:

            test_label = test_label.to(device)
            mask = test_input['attention_mask'].to(device)
            input_id = test_input['input_ids'].squeeze(1).to(device)

            output = model(input_id, mask)
            # print('y_pred',y_pred.size())
            # print(y_pred)
            # print('output',output.size())
            # print(output.argmax(dim=1))
            y_pred = torch.cat((y_pred,output.argmax(dim=1)),0) # Save Prediction

            # print('y_true',y_true.size)
            # print(y_true)
            # print('test_labe',test_label.size())
            # print(test_label)
            y_true = torch.cat((y_true,test_label),0) # Save Truth

            acc = (output.argmax(dim=1) == test_label).sum().item()
            total_acc_test += acc

        print(y_pred.size(),y_true.size())
        metric = BinaryStatScores().to(device)
        print(metric(y_pred, y_true))
    
    print(f'Test Accuracy: {total_acc_test / len(test): .3f}')

In [12]:
print('\n\nCreating the Dataset')
data = CongressHearings(df)
train_data, val_data, test_data = random_split(data, [0.2,0.2,0.6])
print('Lenght of total dataset: ', len(data))
print('Lenght of training data:', len(train_data))



Creating the Dataset
Lenght of total dataset:  611651
Lenght of training data: 122331


In [31]:
from collections import Counter
train_classes = [label for _, label in train_data]
count = Counter(i.item() for i in train_classes)
print(f'Total number of samples: {len(train_data)}\
      \nNumber of samples labeled 0: {count[0]}, {count[0]/len(train_classes)*100: .3f}%\
      \nNumber of samples labeled 1: {count[1]}, {count[1]/len(train_classes)*100: .3f}%')

Total number of samples: 122331      
Number of samples labeled 0: 63102,  51.583%      
Number of samples labeled 1: 59229,  48.417%


In [8]:
# test_df = df[['text_without_stopwords','Party']]
# evaluate(model, CongressHearings(test_df))

torch.Size([611651]) torch.Size([611651])
tensor([     0,      0, 314858, 296793, 296793], device='cuda:0')
Test Accuracy:  0.515
